# Hash Functions and Collisions

```{contents}
:local:
:depth: 2
```

## Dictionary Contracts and Key Equality

```{index} dictionary; key equality; hashing
```

A dictionary associates one value with each distinct key. Looking up a customer ID asks for that ID's record; it does not ask for the first record in order. C# `Dictionary<TKey, TValue>` uses hashing for key-based access. `Add` rejects an existing key, indexer assignment inserts or replaces, and `TryGetValue` reports whether a key exists without treating a missing key as a successful default value.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var owners = new Dictionary<string, string>(StringComparer.OrdinalIgnoreCase);
owners.Add("T-100", "Amina");
owners["t-100"] = "Diego";
Console.WriteLine($"count={owners.Count}, owner={owners["T-100"]}");
Console.WriteLine($"missing={owners.TryGetValue("T-999", out _)}");
Console.WriteLine($"removed={owners.Remove("T-100")}, count={owners.Count}");

This dictionary treats `T-100` and `t-100` as equal, so assignment changes the owner rather than adding another key. Choose the equality policy when creating the collection and apply it consistently. For symbolic ticket IDs, ordinal comparison avoids culture-dependent language rules. This example deliberately ignores case; other applications may need case-sensitive identifiers.

Hashing and equality must agree: equal keys must have equal hash codes. Different keys may have the same hash code, so a lookup must still compare actual keys. A custom comparer supplies both operations. Do not modify fields that determine a stored key's equality or hash code; its new bucket may no longer match the location used during insertion. Strings are immutable, which avoids that particular mutation hazard. [Microsoft's GetHashCode contract](https://learn.microsoft.com/en-us/dotnet/api/system.object.gethashcode) explains this agreement.

## Hash Codes and Bucket Mapping

```{index} hash function; bucket; negative hash code
```

Separate two steps: compute an integer hash code, then map that code into one of m buckets. Hash codes can be negative. C# remainder can also be negative, and `Math.Abs(int.MinValue)` cannot fit in an `int`. Reinterpret the code as an unsigned integer before remainder to obtain a valid index without that overflow.

In [ ]:
using System;
int[] codes = { 12, -1, int.MinValue, int.MaxValue };
int bucketCount = 5;
foreach (int hash in codes)
{
    int bucket = (int)(unchecked((uint)hash) % (uint)bucketCount);
    Console.WriteLine($"hash={hash}, bucket={bucket}");
}

`unchecked((uint)hash)` keeps the original bits and interprets them as a nonnegative number; it does not claim that the code is a unique identifier. BucketCount must be positive. The result lies from zero through m - 1. Equal hash codes always map to the same bucket for a fixed m, but many different codes also map there.

String hash codes may vary across processes or platforms. Use them for temporary placement, not as permanent customer IDs or persisted checksums. The teaching map below uses integer IDs themselves as simple hash codes so its collision trace is reproducible. This is an illustration, not a general recommendation for distributing arbitrary business keys.

## Separate Chaining and Complete Operations

```{index} separate chaining; collision; hash-table update
```

A collision occurs when different keys choose the same bucket. **Separate chaining** stores a collection of entries at each bucket. A lookup computes the bucket and compares keys within that chain. An update must search the chain before appending; otherwise it can create duplicate keys. Removal must preserve the other colliding entries.

With four buckets, IDs 101, 105, and 109 all choose bucket one. To find 109, inspect 101, then 105, then 109. To report that 113 is absent, inspect the entire chain. A successful lookup may stop early; an unsuccessful one must establish that no matching key exists.

The complete `TicketMap` stores integer ticket IDs and string owners. `Put` returns true for a new key and false for replacement. `Remove` reports whether membership changed. `TryGetValue` returns false on absence; ignore its output value in that case.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var owners = new TicketMap();
owners.Put(101, "Amina"); owners.Put(105, "Diego"); owners.Put(109, "Mina");
Console.WriteLine("before=" + owners.DescribeBuckets());
Console.WriteLine($"replacement inserted={owners.Put(105, "Lee")}, count={owners.Count}");
owners.Put(113, "Noor");
Console.WriteLine($"after growth: buckets={owners.BucketCount}, count={owners.Count}");
Console.WriteLine(owners.DescribeBuckets());
owners.Remove(101);
Console.WriteLine($"109 survives={owners.TryGetValue(109, out string owner)}, owner={owner}");
Console.WriteLine($"missing={owners.TryGetValue(101, out _)}, count={owners.Count}");

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
            lines.Add($"{i}: {string.Join(", ", buckets[i].Select(e => e.Id))}");
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}

Each stored entry belongs to the bucket computed from its ID and the current bucket count. Count equals the total number of entries, and each ID occurs once. The internal entries are immutable; replacement installs a new entry with the same ID. An update returns before increasing Count or testing growth because it adds no key.

After removal, another colliding ID remains reachable in the same chain. Our chains are lists, so removal may shift later entries within that chain. The cost depends on chain length rather than the total number of buckets. This map keeps capacity after deletions and has no iterator API; production libraries provide more policies and interfaces.

## Load Factor and Rehashing

```{index} load factor; resizing; rehashing
```

Define load factor α = n / m, where n is the number of entries and m is the number of buckets. Before adding 113, the map has three entries in four buckets, so α = 0.75. A fourth new key would exceed the teaching threshold, so the map doubles to eight buckets before inserting it.

Changing the bucket count changes placement. Copying old bucket lists to the same array indexes is insufficient. Recompute every entry's bucket under the new capacity, a process called **rehashing**.

| ID | Bucket with m = 4 | Bucket with m = 8 |
| --- | ---: | ---: |
| 101 | 1 | 5 |
| 105 | 1 | 1 |
| 109 | 1 | 5 |
| 113 | 1 | 1 |

Growth splits this collision group into two chains. It does not guarantee that all collisions disappear. A poor hash function can keep many keys in one chain even at low load factor.

Under well-distributed hashes and constant-time hashing/equality, chaining operations have expected O(1 + α) work. A pathological chain can require O(n) work. This resize visits all entries and initializes a larger bucket array, giving O(n + m) work for that operation. Geometric growth contributes amortized constant work per new key when load is kept bounded; one growth-triggering insertion can still be linear.

The threshold 0.75 belongs to this teaching map, not to a universal hash-table rule. Separate chaining permits α greater than one. Open addressing needs unused slots and becomes much more expensive as occupancy approaches one. Key hashing itself is an additional cost, such as scanning characters in a long string.

## Open Addressing and Deleted Slots

```{index} open addressing; linear probing; tombstone
```

**Open addressing** stores entries directly in the table. **Linear probing** tries the chosen slot, then subsequent slots with wraparound until it finds the key or a place to insert. A lookup may stop at a slot that has never been occupied, because insertion would have stopped there too. It cannot stop merely because a previous entry was removed.

For capacity seven, IDs 10, 17, and 24 all start at slot three. Linear probing stores them at three, four, and five. Clearing slot three after deleting 10 would make a naive search for 17 stop early. A **tombstone** marks a deleted slot through which lookup must continue.

In [ ]:
using System;
int?[] slots = new int?[7];
slots[3] = 10; slots[4] = 17; slots[5] = 24;
bool[] deleted = new bool[7];
slots[3] = null;
deleted[3] = true;
Console.WriteLine($"naive finds 17={Find(17, false)}");
Console.WriteLine($"tombstone finds 17={Find(17, true)}");
Console.WriteLine($"missing 31={Find(31, true)}");

bool Find(int key, bool honorTombstones)
{
    int index = (int)(unchecked((uint)key) % (uint)slots.Length);
    for (int probes = 0; probes < slots.Length; probes++)
    {
        if (slots[index] == key) return true;
        if (slots[index] is null && (!honorTombstones || !deleted[index])) return false;
        index = (index + 1) % slots.Length;
    }
    return false;
}

Limit a search to at most m probes so a full table cannot cause an endless loop. Insertion can reuse a tombstone, but it must still search far enough to rule out an existing equal key. Otherwise it can add a duplicate before reaching the old copy. Over time, many tombstones lengthen searches; rebuilding the table can remove them.

This example demonstrates a deletion rule, not a complete open-addressed dictionary. The chapter's complete implementation uses chaining. Both strategies must combine placement with actual key equality; neither treats a collision as a duplicate by itself.

## Sets and Frequency Counts

```{index} HashSet; set; frequency count
```

A set stores keys without associated record values. Use `HashSet<T>.Add` to detect a first occurrence, or a dictionary to count occurrences. Give both the same comparer when they must agree about what counts as equal.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

string[] statuses = { "Open", "open", "Closed", "OPEN", "closed" };
var seen = new HashSet<string>(StringComparer.OrdinalIgnoreCase);
var counts = new Dictionary<string, int>(StringComparer.OrdinalIgnoreCase);
foreach (string status in statuses)
{
    seen.Add(status);
    counts.TryGetValue(status, out int count);
    counts[status] = count + 1;
}
Console.WriteLine($"distinct statuses={seen.Count}");
foreach (string status in new[] { "open", "closed" })
    Console.WriteLine($"{status}: {counts[status]}");

The set has two distinct statuses, and the counts are open three and closed two. The example prints keys in an explicit order; a hash collection's enumeration order is not a sorting contract. Sort keys separately when a report requires an order. Neither a set nor a dictionary supplies the priority-selection operation of a heap.

## Exercise

```{index} dictionary; exercise
```

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Exercise: Count status updates with case-insensitive equality.
string[] statuses = { "Waiting", "waiting", "Open", "WAITING" };
var counts = new Dictionary<string, int>(StringComparer.OrdinalIgnoreCase);
// Your code starts here.
/* TODO: update counts for every status. */
// Your code ends here.
Console.WriteLine($"waiting={counts.GetValueOrDefault("waiting")}, open={counts.GetValueOrDefault("open")}");

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// Solution
string[] statuses = { "Waiting", "waiting", "Open", "WAITING" };
var counts = new Dictionary<string, int>(StringComparer.OrdinalIgnoreCase);
foreach (string status in statuses)
{
    counts.TryGetValue(status, out int count);
    counts[status] = count + 1;
}
Console.WriteLine($"waiting={counts["waiting"]}, open={counts["open"]}");

waiting=3, open=1


```{rubric} Footnotes
```

[^1]: The teaching map uses integer keys and list-based chains. It illustrates dictionary contracts and rehashing without reproducing the internal implementation of .NET Dictionary.